In [3]:
import os
import sys
import time
import re
import unicodedata
from datetime import datetime
from typing import List, Dict
from loguru import logger
from pymongo import MongoClient
from pymongo import UpdateOne, UpdateMany

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../.."))

sys.path.append(PROJECT_ROOT)

from constants import MongoDBConfig, MongoDBCollectionConfig, MigrateConfig

# Kết nối MongoDB
client = MongoClient(
    host=MongoDBConfig.HOST, 
    port=MongoDBConfig.PORT,
    username=MongoDBConfig.USERNAME,
    password=MongoDBConfig.PASSWORD
)
# database name
db = client["v03_core_281125"]
articles_col = db["law_articles"]
docs_col = db["law_documents"]

add these fields to law_articles_collection:
  "article_expiry_date": "",
  "article_effective_date": "",
  "article_effective_status": "",
with value in law_documents_collection:
doc_effective_status 
doc_effective_date
doc_expiry_date
find all articles in law_articles_collection, get doc_id and check those fields in law_documents_collection

In [ ]:
unique_doc_ids = articles_col.distinct("doc_id")

logger.info("unique_doc_ids", count=len(unique_doc_ids))

count_updated = 0

# 3. Iterate through each unique document ID
for doc_id in unique_doc_ids:

    parent_doc = docs_col.find_one({"doc_id": doc_id})
    
    if parent_doc:
        # Extract the required fields, defaulting to None or "" if they don't exist
        effective_status = parent_doc.get("doc_effective_status", "")
        effective_date = parent_doc.get("doc_effective_date", "")
        expiry_date = parent_doc.get("doc_expiry_date", "")
        
        # 4. Update all articles that belong to this doc_id
        result = articles_col.update_many(
            {"doc_id": doc_id},
            {
                "$set": {
                    "article_effective_status": effective_status,
                    "article_effective_date": effective_date,
                    "article_expiry_date": expiry_date
                }
            }
        )
        
        count_updated += result.modified_count

logger.info("migration_complete", updated_count=count_updated)

Found 14523 unique documents referenced in articles.
Migration complete. Updated 156102 articles.


In [ ]:
articles = articles_col.find({})
count_updated = 0
for article in articles:
    article_id = article.get('article_id')
    article_effective_status = article.get('article_effective_status', '')
    if article_effective_status == 'Còn hiệu lực':
        articles_col.update_one(
            {"article_id": article_id},
            {
                "$set":{
                    "decree_status_id" : "3969bc0a-a285-4a6d-9865-5b549cf88d20"
                }
            }
        )
        count_updated +=1
    else:
        articles_col.update_one(
            {"article_id" : article_id},
            {
                "$set": {
                    "decree_status_id" : "b04750de-31f5-4266-b5c7-ac56c2bac946"
                }
            }
        )
        count_updated += 1
logger.info("count_updated", count_updated=count_updated)

156102
